## Notebook 1 for data cleaning and merging

In [1]:
# import
import os
import sys
import csv
import re
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

import gzip

In [ ]:

# Dossier contenant les fichiers CSV
folder_path = "Data_spCal"
output_file = "fusion_results.csv.gzip"

# Initialiser une variable pour stocker toutes les lignes extraites
all_data = []
first_file = True

for filename in sorted(os.listdir(folder_path)):
    print(f"Files in process : {filename}")
    if not filename.endswith(".csv"):
        continue

    file_path = os.path.join(folder_path, filename)

    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    try:
        start_idx = next(i for i, line in enumerate(lines) if "# Raw detection data" in line)
        end_idx = next(i for i, line in enumerate(lines) if "# End of export" in line)
    except StopIteration:
        print(f"Missing lines in {filename}, ignored.")
        continue

    block = lines[start_idx + 1:end_idx]  # exclut "# Raw detection data" et "# End of export"

    if first_file:
        if len(block) > 1:
            block = block[:-1]  # Supprimer la ligne juste avant # End of export

        modified_block = []
        for i, line in enumerate(block):
            line = line.strip()
            if i == 0:  # La 3e ligne après "# Raw detection data" est le header
                modified_block.append(line + ",Filename\n")
            else:
                modified_block.append(line + f",\"{filename}\"\n")
        all_data.extend(modified_block)
        first_file = False
    else:
        if len(block) > 3:
            block = block[2:-1]  # Supprimer unités, headers et la dernière ligne
            modified_block = [line.strip() + f",\"{filename}\"\n" for line in block]
            all_data.extend(modified_block)

# Écriture dans le fichier fusionné
with gzip.open(output_file, "wt", encoding="utf-8") as f_out:
    f_out.writelines(all_data)

print(f"Fusion over : {output_file}")

In [ ]:
def clean_col_name(col):
    if col == "U238":
        return "U"
    elif col == "K39":
        return "K"
    elif col not in ['Filename']:
        # Remove trailing digits and letters after the element symbol
        # For example: "Al27" → "Al", "La139" → "La"
        return re.sub(r'[\dA-Z]+$', '', col)
    else:
        return col

In [ ]:
def clean_filename(name):
    # Supprimer l'extension
    name = name.replace('_results.csv', '')
    # Enlever le timestamp ou les chiffres initiaux (facultatif selon ton besoin)
    name = re.sub(r'^\d{2}-\d{2}-\d{2} ?', '', name)  # retire "16-16-57 " ou "20-02-21 "
    # Séparer par "_" et ne garder que la première partie significative
    parts = name.split('_')
    return parts[0] if len(parts) == 1 else ' '.join(parts[:-1])

In [ ]:
# Reading the CSV file with a MultiIndex header
# MultiIndex avec les 2 premières lignes
df = pd.read_csv("fusion_results.csv.gzip", compression='gzip', header=[0, 1])

# Delete columns that contain "counts" in the second level of the MultiIndex
df = df.loc[:, ~df.columns.get_level_values(1).str.contains("counts", case=False, na=False)]

# Keep only the first level of the MultiIndex
df.columns = df.columns.get_level_values(0)

# drop rows with time < 20 seconds
df = df[df['Time'] >= 20]

# drop the Time column
df.drop(columns=["Time"], inplace=True, errors='ignore')

# replace NaN by 0
df.fillna(0, inplace=True)

# Clean column names
df.columns = [clean_col_name(col) for col in df.columns]

# Add a new column before the filename column to sum all elements
df['Sum'] = df.drop(columns=['Filename']).sum(axis=1)
# Put the 'Sum' column just before the 'Filename' column
df = df[[col for col in df.columns if col != 'Filename'] + ['Filename']]

# Change the 'Filename' column to keep only the first part of the filename
df['Filename'] = df['Filename'].apply(clean_filename)

# separate the 'Filename' column into 'Core' and '# sample'
df[['Core', '# sample']] = df['Filename'].str.split(' ', n=1, expand=True)
# Put 0 for blank rows in '# sample'
df['# sample'] = df['# sample'].fillna('0')

# sort the DataFrame by 'Filename' but keep the blank rows first
df.sort_values(by='# sample', key=lambda x: x.str.strip(), inplace=True)
# Reorder the columns to have 'Blank' first

# Reset the index
df.reset_index(drop=True, inplace=True)

# Display the last few rows of the DataFrame
df.tail()

In [ ]:
# drop the column U
df.drop(columns=['U'], inplace=True, errors='ignore')

In [ ]:
# Save the cleaned DataFrame to a new CSV file
df.to_csv("fusion_results_cleaned.csv.gzip", index=False, compression='gzip')

In [ ]:
# import the Data AWI.xlsx as a DataFrame
awi_df = pd.read_excel("Add_data/Data AWI.xls", header=0)
awi_df.head()

In [ ]:
# Clean columns
df.columns = df.columns.str.strip()
awi_df.columns = awi_df.columns.str.strip()

# Ensure merge column exists and is properly formatted
df['# sample'] = df['# sample'].astype(str).str.strip()
awi_df['# sample'] = awi_df['# sample'].astype(str).str.strip()

# Perform merge
merged_df = pd.merge(df, awi_df, on='# sample', how='left')

# Replace 0 values in the '# sample' column with 'Blank'
merged_df['# sample'] = merged_df['# sample'].replace('0', 'Blank')

# sort by depth
merged_df.sort_values(by='Depth', inplace=True)

# remove the row where Depth is > 125
merged_df = merged_df[merged_df['Depth'].astype(float) <= 125]

# remove the rows where Depth is < 105
merged_df = merged_df[merged_df['Depth'].astype(float) >= 105]

# Save and check
merged_df.to_csv("fusion_results_merged.csv.gzip", index=False, compression='gzip')
merged_df.head()

In [ ]:
# display the number of unique # sample in the merged_df
merged_df['# sample'].nunique()

In [ ]:
# display the number of particles in the merged_df
merged_df.shape[0]